# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pratham6306/ml-internship-flyrank/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
!git clone https://github.com/pratham6306/ml-internship-flyrank.git

Cloning into 'ml-internship-flyrank'...
remote: Enumerating objects: 120, done.
remote: Counting objects: 100% (120/120), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 120 (delta 35), reused 100 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (120/120), 1.84 MiB | 11.15 MiB/s, done.
Resolving deltas: 100% (35/35), done.


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

# My Lane as an ML Task

I selected **Lane 2: Refresh / Content Opportunity Scoring**.

This project is primarily a **ranking/scoring** task. The objective is to rank webpages according to their priority for content review rather than simply predicting whether a page is declining. The output is a ranked review queue that helps SEO specialists decide where to spend their limited editorial effort first.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

# Target or Proxy

The final goal is to rank webpages according to their refresh priority.

For the starter dataset, a temporary proxy target is:

**is_declining_label**

This proxy label identifies pages currently marked as declining and is suitable for learning the complete ML workflow. However, it is derived from `trend_direction`, so it should be treated only as a starter proxy rather than an ideal long-term target.

A stronger future target would compare a previous feature window with a later outcome window, such as:

Features from previous 90 days → Decline or recovery during the next 30 days.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

# Success Metric

Since this is a ranking problem, the primary success metric is **Precision@K**.

Precision@K measures how many of the highest-ranked pages actually deserve review.

Additional evaluation metrics may include:

- Average Precision
- Recall
- ROC-AUC (when evaluating the proxy classification model)

These metrics help determine whether the learned ranking improves over a simple rule-based baseline.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [8]:
import pandas as pd

df = pd.read_csv("/content/ml-internship-flyrank/data/raw/content_refresh_anonymized.csv")

# Create the proxy label used by the starter pipeline
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("Dataset Shape:", df.shape)

display(df.head())

print("\nOne row represents one webpage (content item).")

print("\nColumns relevant to this lane:")

display(
    df[
        [
            "content_id",
            "client_id",
            "impressions_90d",
            "clicks_90d",
            "sessions_90d",
            "ctr",
            "avg_position",
            "content_age_days",
            "trend_direction",
            "is_declining_label",
        ]
    ].head()
)

print("\nTarget Distribution:")
print(df["is_declining_label"].value_counts())

Dataset Shape: (30000, 45)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining_label
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,1
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,1



One row represents one webpage (content item).

Columns relevant to this lane:


,content_id,client_id,impressions_90d,clicks_90d,sessions_90d,ctr,avg_position,content_age_days,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,29,17,0.76,10.6,187,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,7,9,0.05,20.3,445,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,11,0.09,36.5,141,down,1
3,content_331d6c4de07b,client_19581e27de,11751,58,78,0.49,6.2,463,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,24,145,0.13,44.0,263,down,1



Target Distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64


### Unit of Analysis

Each row in the dataset represents one pseudonymized webpage (content item).

The observable signals describe historical search and engagement performance for that webpage over the previous 90 days.

These page-level observations form the basis for ranking content review opportunities.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

# Why ML Beats a Fixed Rule

A simple rule such as selecting pages with low CTR or old content is easy to understand but ignores interactions between multiple signals.

Search performance depends on several factors, including impressions, clicks, average position, content age, engagement, and search demand.

Machine learning can combine these observable signals into a more informative ranking while still allowing human reviewers to inspect the final recommendations.

The purpose of the model is to support editorial decisions rather than replace human judgment.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [9]:
print(df["is_declining_label"].value_counts())

display(df[["content_id", "is_declining_label"]].head(10))

is_declining_label
1    16262
0    13738
Name: count, dtype: int64


,content_id,is_declining_label
0,content_304f48230142,1
1,content_a1fb4e703a9e,1
2,content_9aa793d4d895,1
3,content_331d6c4de07b,0
4,content_d99b7a2d90ca,1
5,content_d4084a4bc775,1
6,content_9a34b442b552,1
7,content_a63219c6e95a,0
8,content_5e6c160719bc,1
9,content_c27558df2b0c,1
